In [ ]:
# 셀 1: 패키지 설치
!pip install -q ddgs requests pillow matplotlib ImageHash torch transformers

In [ ]:
# 셀 2: 수집 코드와 같은 기준으로 세 종류를 각 5장 미리보기
import importlib.util
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import matplotlib.pyplot as plt

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "code/phase1a_images/image_crawl.py").is_file())
spec = importlib.util.spec_from_file_location("image_crawl", root / "code/phase1a_images/image_crawl.py")
crawl = importlib.util.module_from_spec(spec)
spec.loader.exec_module(crawl)
processor, model, text_features, device = crawl.load_clip(root)
fig, axes = plt.subplots(3, 5, figsize=(18, 11))
for ax in axes.flat:
    ax.axis("off")
seen = set()
with ThreadPoolExecutor(max_workers=crawl.WORKERS) as pool:
    for row, label in enumerate(crawl.CLASSES):
        results = list(crawl.DDGS(timeout=20).images(crawl.QUERIES[label][0] + " -collage -collection", max_results=100))
        count = 0
        attempted = set()
        for start in range(0, len(results), crawl.BATCH_SIZE):
            candidates = []
            for result in results[start:start + crawl.BATCH_SIZE]:
                url = result.get("image")
                if url and url not in seen and url not in attempted:
                    attempted.add(url)
                    candidates.append(result)
            batch = [item for item in pool.map(crawl.download, candidates) if item is not None]
            if not batch:
                continue
            scores = crawl.score_images([item[1] for item in batch], processor, model, text_features, device)
            for (result, image, _), score in zip(batch, scores):
                if score.argmax().item() != row or score[row].item() < crawl.CLIP_THRESHOLD:
                    continue
                axes[row, count].imshow(image)
                axes[row, count].set_title(f"{label} {count + 1}")
                seen.add(result["image"])
                count += 1
                if count == 5:
                    break
            if count == 5:
                break
        print(f"{label}: {count}/5장 통과")
plt.tight_layout()
plt.show()
